# AlphaTrain Pillar 2U: Pure Policy Distillation

**THE PIVOT:** Drop the value head entirely. All 13M params focused on policy.

We caused a 300-point policy regression (1,244→943) by increasing val_weight
from 0.01 to 1.0 across iterations. The backbone conflict destroyed policy
quality while the value head never learned.

- **val_weight=0, rank_weight=0** — pure policy distillation
- V6 data (1,654 games, 5.4M states, teacher mean=6,971)
- Warm start from **2P ep6** (best policy: 50.5% top-1 accuracy)
- 10 epochs, eval bracket at 3/6/9

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_v6_density_g098_ms200.pt.gz` — V6 tensor (policy targets same in any encoding)
3. `pillar2p_ep6.pt` — best policy model (on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy best POLICY model (2P ep6 — before val_weight destroyed it)
os.makedirs('/content/alphatrain/data', exist_ok=True)
print('Copying best policy model (2P ep6)...')
shutil.copy(f'{DRIVE}/pillar2p_ep6.pt', '/content/alphatrain/data/model.pt')
assert os.path.exists('/content/alphatrain/data/model.pt'), 'Model copy failed!'
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

# Decompress V6 data (policy targets are the same regardless of value encoding)
print('Decompressing V6 data...')
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v6_density_g098_ms200.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

# VERIFY
for f in ['model.pt', 'selfplay.pt']:
    path = f'/content/alphatrain/data/{f}'
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'  OK: {f} ({os.path.getsize(path)/1e6:.0f} MB)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2U: Pure Policy Distillation — no value head, no ranking
# Warm start from 2P epoch 6 (best policy checkpoint)
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --gpu-data --amp --compile \
    --value-bins 128 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.0 --rank-weight 0.0 \
    --resume alphatrain/data/model.pt --warm-start \
    --epochs 10 --batch-size 12288 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2u_best.pt \
    --save-dir /content/checkpoints/pillar2u

In [ ]:
# Copy ALL epoch checkpoints to Drive for bracket eval
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar2u/epoch_*.pt')):
    dst = f'{DRIVE}/pillar2u_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2u/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/pillar2u_{f}')